# vzviz: NISAR Dashboard Demo

This notebook demonstrates how to use `vzviz` to analyze and visualize
the chunk manifest of a remote NISAR HDF5 file.

Run with [juv](https://github.com/manzt/juv):
```bash
juv run examples/nisar_dashboard.ipynb
```

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "earthaccess",
#     "virtualizarr[hdf] @ git+https://github.com/maxrjones/VirtualiZarr@c-dtype",
#     "vzviz @ git+https://github.com/virtual-zarr/vzviz",
#     "obspec-utils @ git+https://github.com/virtual-zarr/obspec-utils@store-loop",
#     "aiohttp",
#     "pandas",
#     "jupyterlab",
#     "holoviews",
#     "panel",
#     "bokeh",
# ]
# ///

## Setup

First, we authenticate with NASA Earthdata and create a ManifestStore from a NISAR file.

In [ ]:
from urllib.parse import urlparse

import earthaccess
import virtualizarr as vz
import vzviz
import holoviews as hv
import panel as pn

from obspec_utils.registry import ObjectStoreRegistry
from obspec_utils.stores import AiohttpStore

# Enable holoviews/bokeh extension for notebooks
hv.extension("bokeh")
pn.extension()

In [ ]:
# Authenticate with NASA Earthdata
earthaccess.login()

In [ ]:
# Query for NISAR data
query = earthaccess.DataGranules()
query.short_name("NISAR_L2_GCOV_BETA_V1")
query.params["attribute[]"] = "int,FRAME_NUMBER,77"
query.params["attribute[]"] = "int,TRACK_NUMBER,5"
results = query.get_all()
print(f"Found {len(results)} granules")

# Get the HTTPS URL
https_links = earthaccess.results.DataGranule.data_links(results[0], access="external")
https_url = https_links[0]
print(f"URL: {https_url}")

In [ ]:
# Parse URL and get auth token
parsed = urlparse(https_url)
base_url = f"{parsed.scheme}://{parsed.netloc}"
token = earthaccess.get_edl_token()["access_token"]

# Create store with authentication
store = AiohttpStore(
    base_url,
    headers={"Authorization": f"Bearer {token}"},
)
registry = ObjectStoreRegistry({base_url: store})

In [ ]:
# Create ManifestStore by parsing the HDF5 file
parser = vz.parsers.HDFParser()
manifest_store = parser(https_url, registry=registry)
print("ManifestStore created!")

## Variables Overview

Get a summary of all variables in the manifest, including shapes, chunk sizes, and storage statistics.

In [ ]:
overview = vzviz.variables_overview(manifest_store)
overview[
    [
        "variable",
        "shape",
        "chunks",
        "dtype",
        "total_chunks",
        "chunk_bytes_human",
        "total_bytes_human",
    ]
]

In [ ]:
# Get store-level statistics
info = vzviz.get_store_info(manifest_store)
print(f"Variables:    {info['n_variables']}")
print(f"Groups:       {info['n_groups']}")
print(f"Total chunks: {info['total_chunks']}")
print(f"Unique files: {info['unique_files']}")

## Select a Variable for Detailed Analysis

Let's find a multi-dimensional variable to analyze in detail.

In [ ]:
# List all variables
variables = vzviz.list_variables(manifest_store)
print(f"Total variables: {len(variables)}")
print("\nFirst 15 variables:")
for var in variables[:15]:
    print(f"  - {var}")

In [ ]:
# Find a good multi-dimensional variable to analyze
target_var = None
for var in variables:
    try:
        arr = vzviz.get_array(manifest_store, var)
        if len(arr.shape) >= 2 and arr.shape[0] > 1 and arr.shape[1] > 1:
            target_var = var
            print(f"Selected: {var}")
            print(f"  Shape: {arr.shape}")
            print(f"  Chunks: {arr.chunks}")
            break
    except Exception:
        continue

if not target_var:
    # Fallback to first variable with chunks
    target_var = variables[0]
    print(f"Using first variable: {target_var}")

## Chunk Grid Info

Detailed information about the chunk grid for the selected variable.

In [ ]:
grid_info = vzviz.chunk_grid_info(manifest_store, target_var)

print(f"Variable: {target_var}")
print(f"Shape:           {grid_info.shape}")
print(f"Chunks:          {grid_info.chunks}")
print(f"Chunk grid:      {grid_info.chunk_grid_shape}")
print(f"Total chunks:    {grid_info.total_chunks}")
print(f"Chunk bytes:     {grid_info.chunk_bytes_human}")
print(f"Total bytes:     {grid_info.total_bytes_human}")
print(f"Dtype:           {grid_info.dtype}")

print("\nDimensions:")
for dim in grid_info.dimensions:
    print(
        f"  Dim {dim.dim_index}: size={dim.size}, chunk_size={dim.chunk_size}, n_chunks={dim.n_chunks}"
    )

## Query Simulation (vischunk-inspired)

Simulate data access patterns and analyze performance metrics like read amplification and efficiency.

In [ ]:
# Simulate a query for the first 10% of data along each dimension
query = {}
for i, size in enumerate(grid_info.shape):
    query_size = max(1, size // 10)
    query[i] = slice(0, query_size)

print(f"Query: {query}")
metrics = vzviz.simulate_query(manifest_store, target_var, query)

print("\nPerformance Metrics:")
print(f"  Array Elements Requested: {metrics.requested_cells:,}")
print(f"  Array Elements Read:      {metrics.cells_read:,}")
print(f"  Read Amplification:       {metrics.read_amplification:.2f}x")
print(f"  Read Efficiency:          {metrics.read_efficiency:.1f}%")
print(f"  Chunks Touched:           {metrics.chunks_touched} / {metrics.total_chunks}")
print(f"  Range Reads:              {metrics.range_reads}")
print(f"  Coalescing Factor:        {metrics.coalescing_factor:.2f}x")
print(f"  Storage Alignment:        {metrics.storage_alignment:.2f}")

In [ ]:
# Compare different query patterns
queries = [
    {0: slice(0, grid_info.shape[0] // 10)},  # First dimension only
    {-1: slice(0, grid_info.shape[-1] // 10)},  # Last dimension only
]
names = ["first_dim_10%", "last_dim_10%"]

# Add single slice query if multi-dimensional
if len(grid_info.shape) >= 2:
    queries.append({0: slice(0, 1)})
    names.append("first_slice")

comparison = vzviz.compare_queries(manifest_store, target_var, queries, names)
comparison[
    [
        "name",
        "requested_cells",
        "cells_read",
        "read_amplification",
        "read_efficiency_pct",
        "chunks_touched",
    ]
]

## Visualizations

### ByteMap

Visualize where chunks are located within each file. Each segment shows a chunk's byte range, colored by variable.

In [ ]:
vzviz.byte_range_chart(manifest_store, target_var, backend="holoviews", width=800)

### ChunkMap

2D grid showing chunks in array index space. Each rectangle spans the array indices contained in that chunk, colored by variable.

In [ ]:
vzviz.chunk_file_heatmap(manifest_store, target_var, width=600, height=400)

## Summary Statistics

In [ ]:
summary = vzviz.manifest_summary(manifest_store)
summary

In [ ]:
file_stats = vzviz.file_summary(manifest_store)
file_stats[
    ["filename", "chunk_count", "total_bytes_human", "byte_range", "is_contiguous"]
].head(10)

## Interactive Dashboard

Launch a full interactive dashboard combining all visualizations.

**Features:**
- **Variables table**: Select rows to filter chunks in ByteMap. Select one variable to view its ChunkMap.
- **ByteMap**: Shows chunk byte ranges within files, colored by variable.
- **ChunkMap**: Box-select a region to see performance metrics and highlight chunks in ByteMap.
- **Selection panel**: Shows selected chunks count, total size, and performance metrics (read amplification, efficiency, coalescing, storage alignment).

In [ ]:
# Create the dashboard for all variables
dashboard = vzviz.manifest_dashboard(manifest_store)
dashboard

In [ ]:
# Create a dashboard focused on a specific variable with cross-panel selection
dashboard_var = vzviz.manifest_dashboard(manifest_store, variable=target_var)
dashboard_var

### Serve Dashboard in Browser

To serve the dashboard in a separate browser window, uncomment and run:

In [ ]:
# Uncomment to serve in browser:
# dashboard.show()